# 05 - Reproduce Results
**NLP Deliverable 2 — Named Entity Recognition**

This notebook **loads the fitted models from disk** (`fitted_models/`) and evaluates them
exactly as required by the guide:

- **Accuracy** on train and test sets, counting only tokens whose ground-truth tag is not `O`.
- **Confusion matrix** on train and test sets.
- **F-score** (`sklearn.metrics.f1_score`).
- **TINY TEST** predictions printed as `w1/t1 w2/t2 ...` plus its accuracy.

Models evaluated: **CRF v1/v2/v3**, **BiLSTM**, **DistilBERT**.

> Colab-friendly. On CPU it caps the (expensive) train-set evaluation via `MAX_TRAIN_EVAL`
> to keep runtime reasonable — set it to `None` for the full set.

In [ ]:
# === Setup ===
import sys, os
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install -q sklearn-crfsuite seqeval
    DATA_DIR = '.'
else:
    DATA_DIR = '../nlp_d2_data'
FITTED_DIR = 'fitted_models'

import warnings; warnings.filterwarnings('ignore')
import re, pickle
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix
sns.set_theme(style='whitegrid')

import torch
HAS_GPU = torch.cuda.is_available()
MAX_TRAIN_EVAL = None if HAS_GPU else 8000   # cap train-eval on CPU; None = full
print('GPU:', HAS_GPU, '| MAX_TRAIN_EVAL:', MAX_TRAIN_EVAL)

## 1. Data + shared helpers

In [ ]:
def load_data(path):
    df = pd.read_csv(path)
    df['words'] = df['words'].astype(str)
    df['sentence_id'] = df['sentence_id'].astype(int)
    return [list(zip(g['words'].tolist(), g['tags'].tolist()))
            for _, g in df.groupby('sentence_id', sort=True)]

train_sents = load_data(os.path.join(DATA_DIR, 'train_data_ner.csv'))
test_sents  = load_data(os.path.join(DATA_DIR, 'test_data_ner.csv'))
tiny_sents  = load_data(os.path.join(DATA_DIR, 'tiny_test.csv'))

train_eval = train_sents if MAX_TRAIN_EVAL is None else train_sents[:MAX_TRAIN_EVAL]
labels_of = lambda sents: [[t for _, t in s] for s in sents]
y_train, y_test, y_tiny = labels_of(train_eval), labels_of(test_sents), labels_of(tiny_sents)

def flatten(seq): return [x for sub in seq for x in sub]

def non_o_accuracy(yt, yp):
    yt, yp = flatten(yt), flatten(yp)
    idx = [k for k, t in enumerate(yt) if t != 'O']
    return accuracy_score([yt[k] for k in idx], [yp[k] for k in idx])

def weighted_f1(yt, yp):
    yt, yp = flatten(yt), flatten(yp)
    labels = sorted(set(yt) - {'O', '<PAD>'})
    return f1_score(yt, yp, average='weighted', labels=labels, zero_division=0)

def plot_cm(yt, yp, title, cmap='Blues'):
    yt, yp = flatten(yt), flatten(yp)
    labels = sorted(set(yt) - {'O', '<PAD>'})
    cm = confusion_matrix(yt, yp, labels=labels)
    plt.figure(figsize=(8,6))
    sns.heatmap(cm, annot=True, fmt='d', cmap=cmap, xticklabels=labels, yticklabels=labels)
    plt.title(title); plt.ylabel('True'); plt.xlabel('Predicted'); plt.show()

def print_tiny(name, sents, y_pred):
    print(f'=== TINY TEST — {name} ===')
    for s, pred in zip(sents, y_pred):
        print(' '.join(f'{w}/{t}' for (w, _), t in zip(s, pred)))
    print(f'Tiny-test accuracy (non-O): {non_o_accuracy(labels_of(sents), y_pred):.4f}\n')

summary = []   # collected metrics per model

## 2. CRF models (v1, v2, v3)
The feature templates **must match training** (defined identically here).

In [ ]:
def word_shape(w):
    s = re.sub('[A-Z]', 'X', w); s = re.sub('[a-z]', 'x', s)
    return re.sub('[0-9]', 'd', s)

def feat_v1(sent, i):
    w = str(sent[i][0])
    return {'bias':1.0,'w.lower':w.lower(),'w.isupper':w.isupper(),
            'w.istitle':w.istitle(),'w.isdigit':w.isdigit()}

def feat_v2(sent, i):
    w = str(sent[i][0])
    f = {'bias':1.0,'w.lower':w.lower(),'w[-3:]':w[-3:],'w[-2:]':w[-2:],'w[:3]':w[:3],
         'w.isupper':w.isupper(),'w.istitle':w.istitle(),'w.isdigit':w.isdigit(),
         'w.shape':word_shape(w)}
    if i>0:
        p=str(sent[i-1][0]); f.update({'-1:lower':p.lower(),'-1:istitle':p.istitle(),'-1:isupper':p.isupper()})
    else: f['BOS']=True
    if i<len(sent)-1:
        n=str(sent[i+1][0]); f.update({'+1:lower':n.lower(),'+1:istitle':n.istitle(),'+1:isupper':n.isupper()})
    else: f['EOS']=True
    return f

def feat_v3(sent, i):
    w = str(sent[i][0])
    f = {'bias':1.0,'w.lower':w.lower(),'w[-4:]':w[-4:],'w[-3:]':w[-3:],'w[-2:]':w[-2:],
         'w[:2]':w[:2],'w[:3]':w[:3],'w.isupper':w.isupper(),'w.istitle':w.istitle(),
         'w.isdigit':w.isdigit(),'w.shape':word_shape(w),'w.len':len(w),
         'w.has_hyphen':'-' in w,'w.has_digit':any(c.isdigit() for c in w),
         'w.is_punct':not any(c.isalnum() for c in w)}
    for off in (-2,-1,1,2):
        j=i+off
        if 0<=j<len(sent):
            c=str(sent[j][0])
            f.update({f'{off:+d}:lower':c.lower(),f'{off:+d}:istitle':c.istitle(),
                      f'{off:+d}:isupper':c.isupper(),f'{off:+d}:shape':word_shape(c)})
    if i==0: f['BOS']=True
    if i==len(sent)-1: f['EOS']=True
    return f

CRF_FEATS = {'CRF v1':feat_v1, 'CRF v2':feat_v2, 'CRF v3':feat_v3}
CRF_FILES = {'CRF v1':'crf_v1.pkl', 'CRF v2':'crf_v2.pkl', 'CRF v3':'crf_v3.pkl'}

def crf_predict(crf, sents, featfn):
    return crf.predict([[featfn(s,i) for i in range(len(s))] for s in sents])

for name, featfn in CRF_FEATS.items():
    path = os.path.join(FITTED_DIR, CRF_FILES[name])
    if not os.path.exists(path):
        print('skip', name, '(not found)'); continue
    crf = pickle.load(open(path,'rb'))
    yp_tr = crf_predict(crf, train_eval, featfn)
    yp_te = crf_predict(crf, test_sents, featfn)
    yp_ti = crf_predict(crf, tiny_sents, featfn)
    print(f'\n########## {name} ##########')
    print(f'Train: acc(non-O)={non_o_accuracy(y_train,yp_tr):.4f}  F1={weighted_f1(y_train,yp_tr):.4f}')
    print(f'Test : acc(non-O)={non_o_accuracy(y_test, yp_te):.4f}  F1={weighted_f1(y_test, yp_te):.4f}')
    plot_cm(y_train, yp_tr, f'{name} — TRAIN confusion (excl. O)')
    plot_cm(y_test,  yp_te, f'{name} — TEST confusion (excl. O)')
    print_tiny(name, tiny_sents, yp_ti)
    summary.append({'model':name,
                    'train_acc_nonO':non_o_accuracy(y_train,yp_tr),
                    'test_acc_nonO':non_o_accuracy(y_test,yp_te),
                    'train_f1':weighted_f1(y_train,yp_tr),
                    'test_f1':weighted_f1(y_test,yp_te),
                    'tiny_acc':non_o_accuracy(y_tiny,yp_ti)})

## 3. BiLSTM

In [ ]:
import torch.nn as nn
bilstm_path = os.path.join(FITTED_DIR, 'bilstm_model.pth')
map_path    = os.path.join(FITTED_DIR, 'dl_mappings.pkl')
if os.path.exists(bilstm_path) and os.path.exists(map_path):
    m = pickle.load(open(map_path,'rb'))
    w2i, t2i, i2t, HP = m['word_to_ix'], m['tag_to_ix'], m['ix_to_tag'], m['hp']
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    class BiLSTMTagger(nn.Module):
        def __init__(self, V, T, emb, hid, drop):
            super().__init__()
            self.embedding = nn.Embedding(V, emb, padding_idx=0)
            self.lstm = nn.LSTM(emb, hid, batch_first=True, bidirectional=True)
            self.dropout = nn.Dropout(drop)
            self.fc = nn.Linear(hid*2, T)
        def forward(self, x):
            e = self.dropout(self.embedding(x)); h,_ = self.lstm(e); return self.fc(self.dropout(h))

    model = BiLSTMTagger(len(w2i), len(t2i), HP['embedding_dim'], HP['hidden_dim'], HP['dropout']).to(device)
    model.load_state_dict(torch.load(bilstm_path, map_location=device)); model.eval()

    @torch.no_grad()
    def bilstm_predict(sents):
        preds=[]
        for s in sents:
            ids=[w2i.get(w,w2i['<UNK>']) for w,_ in s]
            out=model(torch.tensor(ids).unsqueeze(0).to(device))
            preds.append([i2t.get(int(p),'O') for p in out.argmax(-1).squeeze(0).tolist()])
        return preds

    yp_tr=bilstm_predict(train_eval); yp_te=bilstm_predict(test_sents); yp_ti=bilstm_predict(tiny_sents)
    print('\n########## BiLSTM ##########')
    print(f'Train: acc(non-O)={non_o_accuracy(y_train,yp_tr):.4f}  F1={weighted_f1(y_train,yp_tr):.4f}')
    print(f'Test : acc(non-O)={non_o_accuracy(y_test, yp_te):.4f}  F1={weighted_f1(y_test, yp_te):.4f}')
    plot_cm(y_train, yp_tr, 'BiLSTM — TRAIN confusion (excl. O)', cmap='Greens')
    plot_cm(y_test,  yp_te, 'BiLSTM — TEST confusion (excl. O)', cmap='Greens')
    print_tiny('BiLSTM', tiny_sents, yp_ti)
    summary.append({'model':'BiLSTM','train_acc_nonO':non_o_accuracy(y_train,yp_tr),
                    'test_acc_nonO':non_o_accuracy(y_test,yp_te),'train_f1':weighted_f1(y_train,yp_tr),
                    'test_f1':weighted_f1(y_test,yp_te),'tiny_acc':non_o_accuracy(y_tiny,yp_ti)})
else:
    print('BiLSTM artifacts not found — run 03_Model_DL_BiLSTM.ipynb first.')

## 4. DistilBERT

In [ ]:
dist_dir = os.path.join(FITTED_DIR, 'distilbert_ner')
if os.path.isdir(dist_dir):
    from transformers import AutoTokenizer, AutoModelForTokenClassification
    tok = AutoTokenizer.from_pretrained(dist_dir)
    dmodel = AutoModelForTokenClassification.from_pretrained(dist_dir)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu'); dmodel.to(device).eval()
    id2label = dmodel.config.id2label

    @torch.no_grad()
    def bert_predict(sents, batch_size=32):
        out=[]
        words_only=[[w for w,_ in s] for s in sents]
        for st in range(0,len(words_only),batch_size):
            batch=words_only[st:st+batch_size]
            enc=tok(batch,truncation=True,is_split_into_words=True,padding=True,return_tensors='pt').to(device)
            pred=dmodel(**enc).logits.argmax(-1).cpu().numpy()
            for i in range(len(batch)):
                wids=enc.word_ids(batch_index=i); prev=None; tags=[]
                for j,wid in enumerate(wids):
                    if wid is None or wid==prev: prev=wid; continue
                    tags.append(id2label[int(pred[i][j])]); prev=wid
                if len(tags)<len(batch[i]): tags+=['O']*(len(batch[i])-len(tags))
                out.append(tags[:len(batch[i])])
        return out

    yp_tr=bert_predict(train_eval); yp_te=bert_predict(test_sents); yp_ti=bert_predict(tiny_sents)
    print('\n########## DistilBERT ##########')
    print(f'Train: acc(non-O)={non_o_accuracy(y_train,yp_tr):.4f}  F1={weighted_f1(y_train,yp_tr):.4f}')
    print(f'Test : acc(non-O)={non_o_accuracy(y_test, yp_te):.4f}  F1={weighted_f1(y_test, yp_te):.4f}')
    plot_cm(y_train, yp_tr, 'DistilBERT — TRAIN confusion (excl. O)', cmap='Purples')
    plot_cm(y_test,  yp_te, 'DistilBERT — TEST confusion (excl. O)', cmap='Purples')
    print_tiny('DistilBERT', tiny_sents, yp_ti)
    summary.append({'model':'DistilBERT','train_acc_nonO':non_o_accuracy(y_train,yp_tr),
                    'test_acc_nonO':non_o_accuracy(y_test,yp_te),'train_f1':weighted_f1(y_train,yp_tr),
                    'test_f1':weighted_f1(y_test,yp_te),'tiny_acc':non_o_accuracy(y_tiny,yp_ti)})
else:
    print('DistilBERT not found — run 04_Model_DL_Transformer.ipynb first.')

## 5. Summary comparison

In [ ]:
summary_df = pd.DataFrame(summary).set_index('model').round(4)
display(summary_df)
note = '' if MAX_TRAIN_EVAL is None else f'  (train metrics computed on first {MAX_TRAIN_EVAL} sentences)'
print('Higher is better.' + note)